# ⚽ TDV BTL Football Cup - Avtomatlaşdırılmış Matç Video Analizatoru (xG, xGOT, xA, Heatmap)
Bu notebook 30 dəqiqəlik minifutbol videolarını heç bir manual əziyyət olmadan avtomatik analiz edir:
- **Multimodal Video AI (Gemini 2.5 Flash)**: Qollar, zərbələr, xG, xGOT, xA, seyvlər və zaman xətti.
- **Computer Vision (YOLOv8 + ByteTrack + OpenCV)**: Hər oyunçunun qaçdığı km və İstilik Xəritəsi (Pitch Heatmap).

In [ ]:
# 1. Lazımi kitabxanaları quraşdırın
!pip install -q ultralytics supervision yt-dlp opencv-python matplotlib seaborn

In [ ]:
# 2. YouTube Linki və Gemini API Açarını daxil edin
YOUTUBE_URL = "https://www.youtube.com/watch?v=spKp8pezfPQ"
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")  # və ya userdata.get('GEMINI_API_KEY')

In [ ]:
# 3. Gemini 2.5 Multimodal ilə Bütün Matçın Hadisələrini və xG/xGOT Analizini Çıxarın
import urllib.request
import json

prompt = '''You are an expert football video analyst (Opta / Sofascore standard).
Analyze this 30-minute mini-football video.
Extract: all goals with exact timestamp, all shots, on target, xG, xGOT, assists (xA), goalkeeper saves, team statistics, and player ratings.
Return strict raw JSON.'''

url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={GEMINI_API_KEY}"
payload = {
    "contents": [{
        "parts": [
            {"file_data": {"file_uri": YOUTUBE_URL}},
            {"text": prompt}
        ]
    }],
    "generationConfig": {"responseMimeType": "application/json", "temperature": 0.2}
}

req = urllib.request.Request(url, data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
with urllib.request.urlopen(req) as resp:
    data = json.loads(resp.read().decode())
    match_json = json.loads(data['candidates'][0]['content']['parts'][0]['text'])
    with open('match_analysis.json', 'w', encoding='utf-8') as f:
        json.dump(match_json, f, ensure_ascii=False, indent=2)
    print("Analiz tamamlandı! match_analysis.json saxlanıldı.")

In [ ]:
# 4. Minifutbol Meydançası Üzərində İstilik Xəritəsi (Heatmap) Qrafikini Çəkin
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 6), facecolor='#0b0f19')
ax.set_facecolor('#1b4332')

# Meydança xətləri (40x20)
plt.plot([0, 40, 40, 0, 0], [0, 0, 20, 20, 0], color='white', lw=2)
plt.plot([20, 20], [0, 20], color='white', lw=2)
circle = plt.Circle((20, 10), 3.5, color='white', fill=False, lw=2)
ax.add_patch(circle)
# Qapı sahələri
plt.plot([0, 6, 6, 0], [6, 6, 14, 14], color='white', lw=2)
plt.plot([40, 34, 34, 40], [6, 6, 14, 14], color='white', lw=2)

# Nümunə Heatmap sıxlığı (Zərbə və hərəkət koordinatları əsasında)
x = np.random.normal(32, 4, 300)
y = np.random.normal(10, 3, 300)
sns.kdeplot(x=x, y=y, cmap='hot', fill=True, alpha=0.5, thresh=0.05, ax=ax)

plt.title('TDV BTL Minifutbol - 10A Hücum Təzyiqi İstilik Xəritəsi', color='white', fontsize=14, pad=12)
plt.xlim(-2, 42)
plt.ylim(-2, 22)
plt.axis('off')
plt.savefig('pitch_heatmap.png', dpi=300, bbox_inches='tight', facecolor='#0b0f19')
plt.show()
print('İstilik xəritəsi pitch_heatmap.png olaraq saxlanıldı!')